# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [37]:
# !cat publications.tsv

## Import pandas

We are using the very handy pandas library for dataframes.

In [38]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [43]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,2020-04-09,A framework for identifying regional outbreak ...,Nature Medicine,,NaN,covid-framework,https://www.nature.com/articles/s41591-020-0857-9
1,2020-01-13,Prediction of gestational diabetes based on na...,Nature Medicine,,NaN,gdm,https://www.nature.com/articles/s41591-019-0724-8
2,2020-01-13,Axes of a revolution: challenges and promises ...,Nature Medicine,NaN,NaN,axes,https://www.nature.com/articles/s41591-019-0727-5
3,2020-06-02,Building an international consortium for track...,Nature Medicine,NaN,NaN,covid-ccc,https://www.nature.com/articles/s41591-020-0929-x
4,2020-10-10,A Prediction Model to Prioritize Individuals f...,Med,NaN,NaN,covid-test-pred,https://www.sciencedirect.com/science/article/...
5,NaN,Longitudinal symptom dynamics of COVID-19 infe...,NaN,NaN,NaN,NaN,https://www.medrxiv.org/content/10.1101/2020.0...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [44]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [45]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
#     md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
#     if len(str(item.paper_url)) > 5:
#         md += "\nDownload [here](" + item.paper_url + ")\n" 
#         md += "\n[pdf](" + item.paper_url + ")\n" 
        
#     md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

TypeError: can only concatenate str (not "float") to str

These files are in the publications directory, one directory below where we're working from.

In [42]:
!dir ..\_publications

 Volume in drive C is OS
 Volume Serial Number is EE12-78BE

 Directory of C:\Users\user\Documents\GitHub\hrossman.github.io\_publications

21/11/2020  15:53    <DIR>          .
21/11/2020  15:53    <DIR>          ..
21/11/2020  15:53               264 2020-01-13-axes.md
21/11/2020  15:53               272 2020-01-13-gdm.md
21/11/2020  15:53               312 2020-04-09-covid-framework.md
21/11/2020  15:53               273 2020-06-02-covid-ccc.md
21/11/2020  15:53               312 2020-10-10-covid-test-pred.md
               5 File(s)          1,433 bytes
               2 Dir(s)  118,589,255,680 bytes free


In [26]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

'cat' is not recognized as an internal or external command,
operable program or batch file.
